# DonkeyCar steering with ResNet50

Train a steering regressor on native PID recordings from Generated Road and evaluate it offline. The pretrained MyDeZero ResNet50 backbone stays frozen. A new linear head predicts one normalized steering value from each camera frame.

Use the repository's Python environment and run the cells in order. The first run may download the pretrained weights. The notebook selects the checkpoint on `trace6` and scores `trace7` and `trace8` after loading that checkpoint.

In the initial comparison, the linear head improved from 0.2106 validation MAE at epoch 12 to a best of 0.1664 within 80 epochs. A 64-unit hidden layer reached 0.1627. This run keeps the linear head and uses validation MAE for learning-rate reduction and early stopping.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
from time import perf_counter
from uuid import uuid4

import numpy as np

from donkeycar_app.dataset import SteeringDataset
from mydezero import cuda, no_grad, test_mode
from mydezero.dataloaders import DataLoader
from mydezero.functions import mean_squared_error
from mydezero.layers import Linear
from mydezero.models import ResNet50
from mydezero.optimizers import Adam

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the Project I repository.")

index_dir = root / "data" / "processed" / "donkeycar"
train_index = index_dir / "train.json"
test_index = index_dir / "test.json"
fit_routes = ("trace1", "trace2", "trace3", "trace4", "trace5")
validation_routes = ("trace6",)
test_routes = ("trace7", "trace8")
assert not (set(fit_routes) & set(validation_routes + test_routes))
assert not (set(validation_routes) & set(test_routes))

seed = 42
epochs = 80
image_batch_size = 32
feature_batch_size = 256
learning_rate = 3e-4
lr_patience = 5
lr_factor = 0.5
min_learning_rate = 1e-5
early_stop_patience = 15
min_mae_improvement = 5e-4
sharp_threshold = 0.3
use_gpu = cuda.gpu_enable and cuda.cupy.cuda.runtime.getDeviceCount() > 0
np.random.seed(seed)
if use_gpu:
    cuda.cupy.random.seed(seed)
print("Device", "CUDA" if use_gpu else "CPU")

In [ ]:
fit_data = SteeringDataset(root, train_index, fit_routes)
validation_data = SteeringDataset(root, train_index, validation_routes)
frame, label = fit_data[0]
assert frame.shape == (3, 120, 160) and label.shape == (1,)
print(f"Fit samples {len(fit_data):,}, validation samples {len(validation_data):,}")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6]
run_dir = root / "runs" / "donkeycar" / run_id
checkpoint = root / "models" / "donkeycar" / run_id / "best.npz"
run_dir.mkdir(parents=True)
checkpoint.parent.mkdir(parents=True)
config = {
    "fit_routes": fit_routes,
    "validation_routes": validation_routes,
    "test_routes": test_routes,
    "indices": {
        p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in (train_index, test_index)
    },
    "fit_samples": len(fit_data),
    "validation_samples": len(validation_data),
    "model": "ResNet50 frozen pretrained backbone with a scalar linear head",
    "pretrained_weights": ResNet50.WEIGHTS_PATH.format(50),
    "image_preprocessing": "160x120, CHW float32, mean-centered BGR",
    "seed": seed,
    "epochs": epochs,
    "image_batch_size": image_batch_size,
    "feature_batch_size": feature_batch_size,
    "learning_rate": learning_rate,
    "lr_patience": lr_patience,
    "lr_factor": lr_factor,
    "min_learning_rate": min_learning_rate,
    "early_stop_patience": early_stop_patience,
    "min_mae_improvement": min_mae_improvement,
    "sharp_threshold": sharp_threshold,
    "device": "cuda" if use_gpu else "cpu",
    "checkpoint": checkpoint.relative_to(root).as_posix(),
}
(run_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
print("Run directory", run_dir.relative_to(root))

In [ ]:
def extract_features(model, dataset, label):
    loader = DataLoader(dataset, image_batch_size, shuffle=False, gpu=use_gpu)
    features = np.empty((len(dataset), 2048), dtype=np.float32)
    targets = np.empty((len(dataset), 1), dtype=np.float32)
    offset = 0
    started = perf_counter()
    next_report = 2048
    with test_mode(), no_grad():
        for images, steering in loader:
            end = offset + len(images)
            features[offset:end] = cuda.as_numpy(model.extract_features(images))
            targets[offset:end] = cuda.as_numpy(steering)
            offset = end
            if offset >= next_report or offset == len(dataset):
                print(f"{label} features {offset:,}/{len(dataset):,} in {perf_counter() - started:.1f}s")
                next_report += 2048
    if not np.isfinite(features).all():
        raise ValueError(f"Non-finite {label} features")
    return features, targets


model = ResNet50(pretrained=True)
model.fc6 = Linear(1, in_size=2048)
if use_gpu:
    model.to_gpu()
fit_features, fit_targets = extract_features(model, fit_data, "fit")
validation_features, validation_targets = extract_features(model, validation_data, "validation")

In [ ]:
def predict(head, features):
    with test_mode(), no_grad():
        values = head(cuda.as_cupy(features) if use_gpu else features)
    return np.clip(cuda.as_numpy(values), -1.0, 1.0)


def measure(prediction, target):
    if not np.isfinite(prediction).all():
        raise ValueError("Non-finite steering prediction")
    errors = np.abs(prediction - target)
    sharp = np.abs(target) >= sharp_threshold
    return {
        "samples": int(target.size),
        "mae": float(errors.mean()),
        "sharp_samples": int(sharp.sum()),
        "sharp_mae": float(errors[sharp].mean()) if sharp.any() else None,
        "zero_steering_mae": float(np.abs(target).mean()),
    }


loader = DataLoader(list(zip(fit_features, fit_targets)), feature_batch_size, gpu=use_gpu)
optimizer = Adam(lr=learning_rate).setup(model.fc6)
best_mae = float("inf")
best_epoch = 0
progress_mae = float("inf")
epochs_without_progress = 0
epochs_since_lr_change = 0
stop_reason = "epoch_limit"
with (run_dir / "epochs.jsonl").open("w", encoding="utf-8") as metrics_file:
    for epoch in range(1, epochs + 1):
        weighted_loss = 0.0
        for features, steering in loader:
            loss = mean_squared_error(model.fc6(features), steering)
            model.fc6.cleargrads()
            loss.backward()
            optimizer.update()
            weighted_loss += float(loss.data.item()) * len(features)
        validation = measure(predict(model.fc6, validation_features), validation_targets)
        val_mae = validation["mae"]
        if val_mae < best_mae:
            best_mae, best_epoch = val_mae, epoch
            best_weights = model.fc6.W.data.copy()
            best_bias = model.fc6.b.data.copy()

        if val_mae <= progress_mae - min_mae_improvement:
            progress_mae = val_mae
            epochs_without_progress = 0
            epochs_since_lr_change = 0
        else:
            epochs_without_progress += 1
            epochs_since_lr_change += 1

        epoch_lr = optimizer.lr
        should_stop = epochs_without_progress >= early_stop_patience
        if (
            not should_stop
            and epoch < epochs
            and epochs_since_lr_change >= lr_patience
            and epoch_lr > min_learning_rate
        ):
            optimizer.lr = max(min_learning_rate, epoch_lr * lr_factor)
            epochs_since_lr_change = 0
            print(f"Reducing learning rate to {optimizer.lr:.2g} for epoch {epoch + 1}")

        metrics = {
            "epoch": epoch,
            "train_mse": weighted_loss / len(fit_data),
            "validation": validation,
            "learning_rate": epoch_lr,
            "next_learning_rate": optimizer.lr,
            "epochs_without_progress": epochs_without_progress,
        }
        metrics_file.write(json.dumps(metrics, allow_nan=False) + "\n")
        metrics_file.flush()
        print(f"Epoch {epoch:2d}  train MSE {metrics['train_mse']:.4f}  val MAE {validation['mae']:.4f}")
        if should_stop:
            stop_reason = "early_stopping"
            break

epochs_completed = epoch
print(f"Training finished after {epochs_completed} epochs ({stop_reason})")

model.fc6.W.data = best_weights
model.fc6.b.data = best_bias
temporary_checkpoint = checkpoint.with_name("best.partial.npz")
model.save_weights(temporary_checkpoint)
temporary_checkpoint.replace(checkpoint)
print(f"Saved epoch {best_epoch} with validation MAE {best_mae:.4f}")

In [ ]:
offline_model = ResNet50()
offline_model.fc6 = Linear(1)
offline_model.load_weights(checkpoint)
if use_gpu:
    offline_model.to_gpu()

test_data = SteeringDataset(root, test_index, test_routes)
test_features, test_targets = extract_features(offline_model, test_data, "test")
test_predictions = predict(offline_model.fc6, test_features)
route_labels = np.array([sample["route"] for sample in test_data.samples])
test_report = {
    "checkpoint": checkpoint.relative_to(root).as_posix(),
    "best_epoch": best_epoch,
    "epochs_completed": epochs_completed,
    "stop_reason": stop_reason,
    "validation_mae": best_mae,
    "overall": measure(test_predictions, test_targets),
    "by_route": {
        route: measure(test_predictions[route_labels == route], test_targets[route_labels == route])
        for route in test_routes
    },
}
(run_dir / "test.json").write_text(
    json.dumps(test_report, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(json.dumps(test_report, indent=2))

The test metrics compare predictions with the recorded PID steering. The large-steering subset uses `abs(steering) >= 0.3`. The zero-steering MAE is the baseline from always predicting zero. This experiment measures offline imitation accuracy. Simulator driving is evaluated separately.